# LLM Training Pipeline — Part 6: RLVR / GRPO

Stage 6 of 6, the final notebook in the series. Trains `sft_model.pt` directly (no reward
model, no PPO/DPO checkpoint needed) against a rule-based verifiable reward — constrained
story-ending generation, where the reward is 1 if the continuation mentions a target word
within a token budget, else 0 — using GRPO's group-relative advantage in place of a learned
value function. Produces `grpo_model.pt`.

**How to use this notebook:**
- Read each theory section; keep `docs/llm_training_pipeline_reference.html`
  open in another tab (Section 9) for the full derivation.
- Code and tests are already implemented and verified — run cells top to
  bottom. Answer the **Question** cells yourself.

**Parts:**
1. Verifiable Reward
2. GRPO Core (group-relative advantage)
3. GRPO Training Loop

In [1]:
import time, math, os, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tokenizers import ByteLevelBPETokenizer

import sys
sys.path.insert(0, '../..')
from src.llm_pipeline.model import GPTConfig, GPTModel
from src.llm_pipeline.data import TOPIC_KEYWORDS, load_tinystories
from src.llm_pipeline.rlhf import ppo_clipped_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

CKPT_DIR = "../../data/checkpoints/llm_training_pipeline"
torch.manual_seed(0)

tokenizer = ByteLevelBPETokenizer(
    f"{CKPT_DIR}/tinystories_bpe-vocab.json",
    f"{CKPT_DIR}/tinystories_bpe-merges.txt",
)
EOT_ID = tokenizer.token_to_id('<|endoftext|>')

sft_ckpt = torch.load(f"{CKPT_DIR}/sft_model.pt", weights_only=False)
sft_cfg = sft_ckpt['config']
sft_model = GPTModel(sft_cfg).to(device)
sft_model.load_state_dict(sft_ckpt['model_state_dict'])
sft_model.eval()
BLOCK_SIZE = sft_cfg.block_size
print(f"Loaded sft_model.pt — {sum(p.numel() for p in sft_model.parameters()):,} params")

Device: cuda


Loaded sft_model.pt — 13,817,856 params
